# CatBoost Model

Gradient-boosted decision-tree model using CatBoost's native
categorical feature handling.

Baseline to beat:

**Logistic Regression 5-fold CV ROC-AUC: 0.938096**

In [1]:
import pandas as pd
import numpy as np

from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [2]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

In [3]:
train.shape, test.shape

((668665, 15), (286571, 14))

In [4]:
X = train.drop(columns=["Will_Buy_EV", "id"])

y = train["Will_Buy_EV"].map({
    "No": 0,
    "Yes": 1
})

In [5]:
categorical_features = [
    "Gender",
    "City_Type",
    "Current_Car_Type",
    "Home_Charging_Possible",
    "Subsidy_Available",
    "Range_Anxiety_Level",
]

In [6]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [7]:
catboost_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=7,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    allow_writing_files=False
)

In [8]:
catboost_model.fit(
    X_train,
    y_train,
    cat_features=categorical_features,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=100
)

0:	test: 0.9322243	best: 0.9322243 (0)	total: 267ms	remaining: 4m 27s
100:	test: 0.9396912	best: 0.9396912 (100)	total: 17.5s	remaining: 2m 35s
200:	test: 0.9402398	best: 0.9402398 (200)	total: 32.3s	remaining: 2m 8s
300:	test: 0.9406040	best: 0.9406040 (300)	total: 48s	remaining: 1m 51s
400:	test: 0.9409551	best: 0.9409551 (400)	total: 1m 3s	remaining: 1m 35s
500:	test: 0.9411507	best: 0.9411507 (500)	total: 1m 19s	remaining: 1m 19s
600:	test: 0.9412722	best: 0.9412728 (599)	total: 1m 35s	remaining: 1m 3s
700:	test: 0.9413525	best: 0.9413536 (697)	total: 1m 52s	remaining: 47.9s
800:	test: 0.9413966	best: 0.9414013 (796)	total: 2m 8s	remaining: 31.9s
900:	test: 0.9414230	best: 0.9414238 (894)	total: 2m 24s	remaining: 15.9s
999:	test: 0.9414309	best: 0.9414339 (988)	total: 2m 40s	remaining: 0us

bestTest = 0.9414339188
bestIteration = 988

Shrink model to first 989 iterations.


CatBoostClassifier(allow_writing_files=False, depth=7, eval_metric='AUC', iterations=1000, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=100)

In [9]:
catboost_valid_predictions = catboost_model.predict_proba(
    X_valid
)[:, 1]

In [10]:
catboost_auc = roc_auc_score(
    y_valid,
    catboost_valid_predictions
)

catboost_auc

0.9414339187754719

In [11]:
catboost_model.get_best_iteration()

988

In [12]:
catboost_model.get_best_score()

{'learn': {'Logloss': 0.22170984874315397},
 'validation': {'Logloss': 0.22715474638212077, 'AUC': 0.9414339187754718}}

In [13]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": catboost_model.get_feature_importance()
}).sort_values(
    "Importance",
    ascending=False
)

feature_importance

,Feature,Importance
11,Subsidy_Available,54.166827
6,Environmental_Concern_Level,19.003002
1,Annual_Income_USD,10.311684
12,Range_Anxiety_Level,4.097538
0,Age,3.163689
2,Daily_Commute_km,2.923753
4,Charging_Stations_Near_Home,1.797368
5,Charging_Stations_Near_Work,1.753934
10,Home_Charging_Possible,0.952542
3,Number_of_Cars_Owned,0.721709


In [14]:
from sklearn.model_selection import StratifiedKFold

In [15]:
X_test = test.drop(columns=["id"])

In [16]:
X.shape, X_test.shape

((668665, 13), (286571, 13))

In [17]:
oof_predictions = np.zeros(len(X))
test_predictions = np.zeros(len(X_test))

fold_scores = []
best_iterations = []

In [18]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [19]:
for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):

    print(f"\n{'=' * 50}")
    print(f"FOLD {fold}")
    print(f"{'=' * 50}")

    X_train_fold = X.iloc[train_idx]
    X_valid_fold = X.iloc[valid_idx]

    y_train_fold = y.iloc[train_idx]
    y_valid_fold = y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=2000,
        learning_rate=0.05,
        depth=7,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42 + fold,
        verbose=200,
        allow_writing_files=False
    )

    model.fit(
        X_train_fold,
        y_train_fold,
        cat_features=categorical_features,
        eval_set=(X_valid_fold, y_valid_fold),
        early_stopping_rounds=150
    )

    valid_pred = model.predict_proba(
        X_valid_fold
    )[:, 1]

    test_pred = model.predict_proba(
        X_test
    )[:, 1]

    oof_predictions[valid_idx] = valid_pred

    test_predictions += test_pred / 5

    fold_auc = roc_auc_score(
        y_valid_fold,
        valid_pred
    )

    fold_scores.append(fold_auc)
    best_iterations.append(
        model.get_best_iteration()
    )

    print(f"\nFold {fold} AUC: {fold_auc:.6f}")
    print(
        f"Best iteration: "
        f"{model.get_best_iteration()}"
    )


FOLD 1
0:	test: 0.9310025	best: 0.9310025 (0)	total: 180ms	remaining: 6m
200:	test: 0.9391479	best: 0.9391479 (200)	total: 40.6s	remaining: 6m 3s
400:	test: 0.9399183	best: 0.9399183 (400)	total: 1m 19s	remaining: 5m 17s
600:	test: 0.9402818	best: 0.9402818 (600)	total: 2m	remaining: 4m 40s
800:	test: 0.9404437	best: 0.9404437 (800)	total: 2m 42s	remaining: 4m 2s
1000:	test: 0.9405128	best: 0.9405155 (990)	total: 3m 24s	remaining: 3m 23s
1200:	test: 0.9405297	best: 0.9405355 (1173)	total: 4m 8s	remaining: 2m 45s
1400:	test: 0.9405521	best: 0.9405591 (1353)	total: 4m 45s	remaining: 2m 2s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.9405591038
bestIteration = 1353

Shrink model to first 1354 iterations.

Fold 1 AUC: 0.940559
Best iteration: 1353

FOLD 2
0:	test: 0.9319474	best: 0.9319474 (0)	total: 157ms	remaining: 5m 14s
200:	test: 0.9402341	best: 0.9402341 (200)	total: 30.6s	remaining: 4m 33s
400:	test: 0.9409016	best: 0.9409016 (400)	total: 1m 1s	remaining: 4m

In [20]:
oof_auc = roc_auc_score(
    y,
    oof_predictions
)

print("\nFold scores:")

for fold, score in enumerate(fold_scores, start=1):
    print(f"Fold {fold}: {score:.6f}")

print()
print(f"Mean fold AUC: {np.mean(fold_scores):.6f}")
print(f"OOF AUC:       {oof_auc:.6f}")
print(f"Std AUC:       {np.std(fold_scores):.6f}")

print()
print("Best iterations:")
print(best_iterations)


Fold scores:
Fold 1: 0.940559
Fold 2: 0.941387
Fold 3: 0.942775
Fold 4: 0.942211
Fold 5: 0.941649

Mean fold AUC: 0.941716
OOF AUC:       0.941711
Std AUC:       0.000751

Best iterations:
[1353, 980, 1370, 914, 1243]


In [21]:
test_predictions[:10]

array([9.84569622e-03, 2.09297943e-02, 4.54323524e-03, 2.81906464e-03,
       2.18532041e-02, 4.39356309e-03, 3.57981995e-01, 5.26386289e-05,
       6.57582176e-01, 6.01056348e-03])

In [22]:
test_predictions.min(), test_predictions.max()

(np.float64(4.858386569715281e-07), np.float64(0.9884291144605194))

In [23]:
submission = pd.DataFrame({
    "id": test["id"],
    "Will_Buy_EV": test_predictions
})

In [24]:
submission.head()

,id,Will_Buy_EV
0,668665,0.009846
1,668666,0.020930
2,668667,0.004543
3,668668,0.002819
4,668669,0.021853


In [25]:
submission.shape

(286571, 2)

In [26]:
submission.isna().sum()

id             0
Will_Buy_EV    0
dtype: int64

In [27]:
submission["id"].equals(test["id"])

True

In [28]:
submission.to_csv(
    "../outputs/submissions/submission_001_catboost_cv.csv",
    index=False
)

In [29]:
pd.read_csv(
    "../outputs/submissions/submission_001_catboost_cv.csv"
).head()

,id,Will_Buy_EV
0,668665,0.009846
1,668666,0.020930
2,668667,0.004543
3,668668,0.002819
4,668669,0.021853


In [30]:
from pathlib import Path

prediction_dir = Path("../outputs/predictions")
prediction_dir.mkdir(parents=True, exist_ok=True)

In [31]:
oof_df = pd.DataFrame({
    "id": train["id"],
    "target": y,
    "catboost_oof": oof_predictions
})

oof_df.to_csv(
    prediction_dir / "catboost_oof.csv",
    index=False
)

In [32]:
test_pred_df = pd.DataFrame({
    "id": test["id"],
    "catboost_prediction": test_predictions
})

test_pred_df.to_csv(
    prediction_dir / "catboost_test.csv",
    index=False
)

## Results

- Mean 5-fold CV ROC-AUC: **0.941716**
- OOF ROC-AUC: **0.941711**
- CV standard deviation: **0.000751**
- Public Kaggle leaderboard ROC-AUC: **0.941560**